# Xenium Bundle Ingest

Lossless conversion of a Xenium output bundle (cells.parquet, transcripts.parquet, morphology.ome.tif, experiment.xenium, ...) into a SpatialData zarr under `data/raw/`.

Driven by env vars so a single SLURM array job can run this for every sample:
- `XENIUM_SAMPLE_ID`  — short id, used as the zarr filename stem (e.g. `0041323`)
- `XENIUM_BUNDLE_DIR` — absolute path to the `output-XETG...` directory
- `XENIUM_RAW_DIR`    — destination dir (default `../data/raw`)

Output: `${XENIUM_RAW_DIR}/${XENIUM_SAMPLE_ID}.zarr`

In [ ]:
import os
import shutil
from pathlib import Path

import spatialdata as sd
from spatialdata_io import xenium

SAMPLE_ID = os.environ.get("XENIUM_SAMPLE_ID", "0041323")
BUNDLE_DIR = Path(os.environ.get(
    "XENIUM_BUNDLE_DIR",
    f"/blue/maigan/smith6jt/Xenium_Analysis/data/output-XETG00298__{SAMPLE_ID}__Region_1__20260416__180001",
))
RAW_DIR = Path(os.environ.get("XENIUM_RAW_DIR", "../data/raw"))
ZARR_PATH = RAW_DIR / f"{SAMPLE_ID}.zarr"

RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Sample:    {SAMPLE_ID}")
print(f"Bundle:    {BUNDLE_DIR}")
print(f"Zarr out:  {ZARR_PATH.resolve()}")

if not BUNDLE_DIR.exists():
    raise FileNotFoundError(f"Bundle directory not found: {BUNDLE_DIR}")

# Skip the 35GB morphology TIFFs and aligned images — downstream analysis uses only the
# cells_table AnnData and (optionally) boundaries/labels for visualization. Keeping
# transcripts + polygons + rasterized cell labels; dropping whole-tissue images.
XENIUM_KWARGS = dict(
    cells_boundaries=True,
    nucleus_boundaries=True,
    cells_labels=True,
    nucleus_labels=True,
    transcripts=True,
    morphology_mip=False,
    morphology_focus=False,
    aligned_images=False,
    cells_table=True,
    n_jobs=int(os.environ.get("XENIUM_N_JOBS", 4)),
)

if ZARR_PATH.exists():
    print(f"Zarr already exists; verifying it loads ...")
    try:
        sdata = sd.read_zarr(ZARR_PATH)
        print(f"Loaded existing zarr ({list(sdata.tables.keys())} tables); skipping re-import.")
    except Exception as err:
        print(f"Existing zarr is unreadable ({err!r}); recreating ...")
        shutil.rmtree(ZARR_PATH)
        sdata = xenium(BUNDLE_DIR, **XENIUM_KWARGS)
        sdata.write(ZARR_PATH)
else:
    print("Running spatialdata_io.xenium(...) ...")
    sdata = xenium(BUNDLE_DIR, **XENIUM_KWARGS)
    sdata.write(ZARR_PATH)
    print(f"Wrote {ZARR_PATH}")

adata = sdata.tables["table"]
print(f"Table shape: {adata.shape}")
print(f"obs columns: {list(adata.obs.columns)}")
print(f"obsm keys:   {list(adata.obsm.keys())}")